In [3]:
import os
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

benchmark_dir = '/content/drive/MyDrive/benchmark'
packages_dir = os.path.join(benchmark_dir, 'packages')
songs_dir = os.path.join(benchmark_dir, 'songs')
result_dir = os.path.join(benchmark_dir, 'result')

os.makedirs(packages_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

if packages_dir not in sys.path:
    sys.path.insert(0, packages_dir)

print("Benchmark directory:", benchmark_dir)
print("Songs directory:", songs_dir)
print("Result directory:", result_dir)


Mounted at /content/drive
Benchmark directory: /content/drive/MyDrive/benchmark
Songs directory: /content/drive/MyDrive/benchmark/songs
Result directory: /content/drive/MyDrive/benchmark/result


In [4]:
import importlib.util

def install_if_missing(package, import_name=None):
    if import_name is None:
        import_name = package

    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package}...")
        !pip install -q --target={packages_dir} --no-deps {package}
    else:
        print(f"{package}: already installed")


install_if_missing("openunmix")
install_if_missing("stempeg")
install_if_missing("ffmpeg-python", "ffmpeg")

openunmix: already installed
stempeg: already installed
ffmpeg-python: already installed


In [5]:
import time
import gc

import torch
import torchaudio
import openunmix
import stempeg
import ffmpeg

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

separator = openunmix.umx(
    device=device,
    pretrained=True
)

separator.eval()

print(f"Open-Unmix ready | device: {device}")

Downloading: "https://zenodo.org/records/3370486/files/vocals-c8df74a5.pth" to /root/.cache/torch/hub/checkpoints/vocals-c8df74a5.pth


100%|██████████| 34.0M/34.0M [00:06<00:00, 5.23MB/s]


Downloading: "https://zenodo.org/records/3370486/files/drums-5a48008b.pth" to /root/.cache/torch/hub/checkpoints/drums-5a48008b.pth


100%|██████████| 34.0M/34.0M [00:01<00:00, 22.8MB/s]


Downloading: "https://zenodo.org/records/3370486/files/bass-646024d3.pth" to /root/.cache/torch/hub/checkpoints/bass-646024d3.pth


100%|██████████| 34.0M/34.0M [00:01<00:00, 23.8MB/s]


Downloading: "https://zenodo.org/records/3370486/files/other-f8e132cc.pth" to /root/.cache/torch/hub/checkpoints/other-f8e132cc.pth


100%|██████████| 34.0M/34.0M [00:04<00:00, 7.83MB/s]

Open-Unmix ready | device: cuda


In [7]:
songs = sorted(
    [
        os.path.join(songs_dir, file)
        for file in os.listdir(songs_dir)
        if file.endswith(".stem.mp4")
    ]
)

print(f"Found {len(songs)} songs")

if len(songs) != 50:
    print("WARNING: Expected 50 songs!")

Found 50 songs


In [11]:
source_names = ["vocals", "drums", "bass", "other"]

for track_number, stem_file in enumerate(songs[40:50], start=41):

    track_name = f"track_{track_number:03d}"
    output_dir = os.path.join(result_dir, track_name)

    os.makedirs(output_dir, exist_ok=True)

    print(f"\n[{track_name}] {os.path.basename(stem_file)}")

    # Load mixture
    mixture, rate = stempeg.read_stems(
        stem_file,
        stem_id=0,
        sample_rate=44100,
        multiprocess=False
    )

    # Convert to PyTorch tensor
    audio = (
        torch.from_numpy(mixture.T)
        .unsqueeze(0)
        .float()
        .to(device)
    )

    # Inference
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        estimates = separator(audio)

    if device == "cuda":
        torch.cuda.synchronize()

    inference_time = time.perf_counter() - start

    # Save separated sources
    for i, name in enumerate(source_names):

        output = estimates[0, i].cpu()

        output_path = os.path.join(
            output_dir,
            f"{name}.wav"
        )

        torchaudio.save(
            output_path,
            output,
            44100
        )

    duration = mixture.shape[0] / rate
    rtf = inference_time / duration

    print(f"Duration: {duration:.2f} s")
    print(f"Inference: {inference_time:.2f} s")
    print(f"RTF: {rtf:.4f}")

    # Free memory before next song
    del mixture
    del audio
    del estimates

    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()


[track_041] The Easton Ellises (Baumi) - SDRNR.stem.mp4
Duration: 234.50 s
Inference: 5.47 s
RTF: 0.0233

[track_042] The Easton Ellises - Falcon 69.stem.mp4
Duration: 246.80 s
Inference: 5.70 s
RTF: 0.0231

[track_043] The Long Wait - Dark Horses.stem.mp4
Duration: 305.53 s
Inference: 7.92 s
RTF: 0.0259

[track_044] The Mountaineering Club - Mallory.stem.mp4
Duration: 244.27 s
Inference: 6.54 s
RTF: 0.0268

[track_045] The Sunshine Garcia Band - For I Am The Moon.stem.mp4
Duration: 320.11 s
Inference: 8.44 s
RTF: 0.0264

[track_046] Timboz - Pony.stem.mp4
Duration: 252.87 s
Inference: 6.46 s
RTF: 0.0256

[track_047] Tom McKenzie - Directions.stem.mp4
Duration: 175.66 s
Inference: 4.14 s
RTF: 0.0235

[track_048] Triviul feat. The Fiend - Widow.stem.mp4
Duration: 234.92 s
Inference: 5.71 s
RTF: 0.0243

[track_049] We Fell From The Sky - Not You.stem.mp4
Duration: 207.70 s
Inference: 5.50 s
RTF: 0.0265

[track_050] Zeno - Signs.stem.mp4
Duration: 234.20 s
Inference: 6.29 s
RTF: 0.0269
